# 02_block_cam_sensitivity_mel_nv

Qualitative block sensitivity check for MEL vs NV CAMs.

Goal:
- Compare where heatmaps focus across transformer blocks.
- Use this as an internal visual analysis notebook.
- Notebook 03 can then do quantitative block analysis.

Default blocks:
`[-1, -2, -4, -6, -8, -10, -12]`

Output design:
- Generate panels for CLS and GAP separately.
- Build one PDF per model.
- Each PDF page = one image.
- Each page has one row per block.
- Columns: RGB lesion outline, Grad CAM target, Diff CAM, Finer CAM.


In [1]:
from pathlib import Path
import json
import subprocess
import shlex
import pandas as pd

# =============================================================================
# 1. MAIN PARAMETERS
# =============================================================================

# Notebook location: notebooks/mel_nv/02_block_cam_sensitivity_mel_nv.ipynb
REPO_ROOT = Path("../..").resolve()
HAM_ROOT = REPO_ROOT / "data" / "HAM10000"
MEL_NV_ROOT = HAM_ROOT / "mel_nv"

IMG_DIR = HAM_ROOT
MASK_ROOT = HAM_ROOT

# Keep qualitative first. all_test with 7 blocks x 2 models is heavy.
# Recommended: qualitative
SAMPLE_MODE = "qualitative"  # "qualitative" or "all_test"

# Requested qualitative block sweep
TARGET_BLOCK_INDICES = [-1, -2, -4, -6, -8, -10, -12]

# Set True first if you only want to inspect commands.
DRY_RUN = False

# If None, use all rows in the selected CSV.
# For faster debugging, set e.g. NUM_SAMPLES_OVERRIDE = 2
NUM_SAMPLES_OVERRIDE = None

# For block sensitivity, keep columns compact.
# Full version possible: rgb_gt_mask,gradcam_a,gradcam_b,map_diff,finercam
PANEL_ITEMS = "rgb_gt_mask,gradcam_a,map_diff,finercam"
PANEL_SUFFIX = PANEL_ITEMS.replace(",", "_")

# Finer-CAM comparison strength
ALPHA = 0.8

# Paths
CLEAN_CSV = MEL_NV_ROOT / "ham_mel_nv_clean.csv"
QUAL_CSV = MEL_NV_ROOT / "ham_mel_nv_clean_qualitative_10_per_class_seed42.csv"

CLS_CKPT = REPO_ROOT / "external" / "checkpoints4" / "checkpoint-best-cls.pth"
GAP_CKPT = REPO_ROOT / "external" / "checkpoints4" / "checkpoint-best-gap.pth"

GT_COL = "gt_label"
CLASS_ARGS = ["--class_names", "MEL,NV"]
COMPARE_ARGS = [
    "--compare_mode", "gt_pair",
    "--A", "MEL",
    "--B", "NV",
    "--topk_compare", "1",
]

SCENARIOS = [
    {
        "name": "CLS HA 0.5",
        "short_name": "cls_ha05",
        "checkpoint": CLS_CKPT,
        "checkpoint_model_type": "panderm",
        "pooling": "cls",
    },
    {
        "name": "GAP HA 0.25",
        "short_name": "gap_ha025",
        "checkpoint": GAP_CKPT,
        "checkpoint_model_type": "panderm",
        "pooling": "mean",
    },
]

OUT_ROOT = REPO_ROOT / "outputs" / "mel_nv" / f"block_cam_sensitivity_{SAMPLE_MODE}"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("OUT_ROOT:", OUT_ROOT)
print("CLEAN_CSV exists:", CLEAN_CSV.exists(), CLEAN_CSV)
print("QUAL_CSV exists:", QUAL_CSV.exists(), QUAL_CSV)
print("CLS_CKPT exists:", CLS_CKPT.exists(), CLS_CKPT)
print("GAP_CKPT exists:", GAP_CKPT.exists(), GAP_CKPT)
print("Blocks:", TARGET_BLOCK_INDICES)

for scenario in SCENARIOS:
    print("\n", scenario["name"])
    print("  checkpoint:", scenario["checkpoint"])
    print("  pooling:", scenario["pooling"])
    if not Path(scenario["checkpoint"]).exists():
        print("  [WARN] missing checkpoint")


REPO_ROOT: /Users/choekyelnyungmartsang/Developer/master-thesis
OUT_ROOT: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/mel_nv/block_cam_sensitivity_qualitative
CLEAN_CSV exists: True /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/mel_nv/ham_mel_nv_clean.csv
QUAL_CSV exists: True /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/mel_nv/ham_mel_nv_clean_qualitative_10_per_class_seed42.csv
CLS_CKPT exists: True /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints4/checkpoint-best-cls.pth
GAP_CKPT exists: True /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints4/checkpoint-best-gap.pth
Blocks: [-1, -2, -4, -6, -8, -10, -12]

 CLS HA 0.5
  checkpoint: /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints4/checkpoint-best-cls.pth
  pooling: cls

 GAP HA 0.25
  checkpoint: /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints4/checkpoint-best-gap.pth
  poolin

## 2. Build active CSV


In [2]:
def safe_name(name: str) -> str:
    out = str(name).lower()
    for ch in [" ", "/", "\\", ":", ";", ",", "(", ")", "[", "]", "{", "}", "+", "."]:
        out = out.replace(ch, "_")
    while "__" in out:
        out = out.replace("__", "_")
    return out.strip("_")


def make_active_csv() -> tuple[Path, int, pd.DataFrame]:
    csv_out_dir = OUT_ROOT / "csv"
    csv_out_dir.mkdir(parents=True, exist_ok=True)

    if SAMPLE_MODE == "qualitative":
        active_csv = QUAL_CSV
        df = pd.read_csv(active_csv)
    elif SAMPLE_MODE == "all_test":
        df = pd.read_csv(CLEAN_CSV)
        df = df[df["split"].astype(str).str.lower().eq("test")].copy()
        df = df.sort_values(["gt_label", "image_id"]).reset_index(drop=True)
        active_csv = csv_out_dir / "ham_mel_nv_clean_all_test.csv"
        df.to_csv(active_csv, index=False)
    else:
        raise ValueError("SAMPLE_MODE must be 'qualitative' or 'all_test'.")

    if NUM_SAMPLES_OVERRIDE is None:
        num_samples = len(df)
    else:
        num_samples = min(int(NUM_SAMPLES_OVERRIDE), len(df))

    print("ACTIVE_CSV:", active_csv)
    print("NUM_SAMPLES:", num_samples)
    display(df.head())
    display(df.head(num_samples).groupby(["split", "gt_label"]).size().unstack(fill_value=0))
    return active_csv, num_samples, df.head(num_samples).copy()


ACTIVE_CSV, NUM_SAMPLES, DISPLAY_DF = make_active_csv()


ACTIVE_CSV: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/mel_nv/ham_mel_nv_clean_qualitative_10_per_class_seed42.csv
NUM_SAMPLES: 20


,lesion_id,image_id,image,dx,gt_label,label,label_2class,binary_label,split,image_rel_path,...,cue_applied,cue_mask_rel_path,cue_mode,dx_type,age,sex,localization,dataset,age_group,dx_norm
0,HAM_0005846,ISIC_0024459,ISIC_0024459.jpg,mel,MEL,4,0,0,test,images/ISIC_0024459.jpg,...,False,NaN,clean,histo,80.0,male,back,vienna_dias,old,mel
1,HAM_0007272,ISIC_0024756,ISIC_0024756.jpg,mel,MEL,4,0,0,test,images/ISIC_0024756.jpg,...,False,NaN,clean,histo,60.0,male,lower extremity,rosendahl,old,mel
2,HAM_0002576,ISIC_0025414,ISIC_0025414.jpg,mel,MEL,4,0,0,test,images/ISIC_0025414.jpg,...,False,NaN,clean,histo,55.0,male,lower extremity,rosendahl,old,mel
3,HAM_0007031,ISIC_0025616,ISIC_0025616.jpg,mel,MEL,4,0,0,test,images/ISIC_0025616.jpg,...,False,NaN,clean,histo,70.0,female,upper extremity,rosendahl,old,mel
4,HAM_0006696,ISIC_0026094,ISIC_0026094.jpg,mel,MEL,4,0,0,test,images/ISIC_0026094.jpg,...,False,NaN,clean,histo,20.0,male,back,rosendahl,young,mel


gt_label,MEL,NV
split,,
test,10,10


## 3. Generate CAM panels for all blocks

This calls `scripts.generate_finer_cam_panderm` once per model and block.

Output folder pattern:

`outputs/mel_nv/block_cam_sensitivity_<mode>/panels/<model>/block_<block>/`


In [3]:
def run_command(cmd: list[str], dry_run: bool = False):
    print("\n" + "=" * 100)
    print(" ".join(shlex.quote(str(x)) for x in cmd))
    print("=" * 100)
    if dry_run:
        return
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)


def block_dir_name(block_index: int) -> str:
    return f"block_{block_index}".replace("-", "minus")


def generate_panels():
    panel_root = OUT_ROOT / "panels"
    panel_root.mkdir(parents=True, exist_ok=True)

    for scenario in SCENARIOS:
        for block_index in TARGET_BLOCK_INDICES:
            scenario_out_dir = panel_root / scenario["short_name"] / block_dir_name(block_index)
            scenario_out_dir.mkdir(parents=True, exist_ok=True)

            cmd = [
                "python", "-m", "scripts.generate_finer_cam_panderm",
                "--csv", str(ACTIVE_CSV),
                "--image_col", "image_rel_path",
                "--img_dir", str(IMG_DIR),
                "--gt_col", GT_COL,
                "--checkpoint", str(scenario["checkpoint"]),
                "--checkpoint_model_type", scenario.get("checkpoint_model_type", "panderm"),
                "--pooling", scenario["pooling"],
                "--out_dir", str(scenario_out_dir),
                "--num_samples", str(NUM_SAMPLES),
                "--method", "finercam",
                "--alpha", str(ALPHA),
                "--panel_items", PANEL_ITEMS,
                "--mask_root", str(MASK_ROOT),
                "--mask_col", "mask_rel_path",
                "--target_block_index", str(block_index),
                "--clinician_labels",
                "--model_display_name", f"{scenario['name']} block {block_index}",
                # "--save_json",
                "--save_raw_cams",
            ]
            cmd += CLASS_ARGS
            cmd += COMPARE_ARGS

            print(f"\nGenerating: {scenario['name']} | block {block_index}")
            run_command(cmd, dry_run=DRY_RUN)


generate_panels()



Generating: CLS HA 0.5 | block -1

python -m scripts.generate_finer_cam_panderm --csv /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/mel_nv/ham_mel_nv_clean_qualitative_10_per_class_seed42.csv --image_col image_rel_path --img_dir /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000 --gt_col gt_label --checkpoint /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints4/checkpoint-best-cls.pth --checkpoint_model_type panderm --pooling cls --out_dir /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/mel_nv/block_cam_sensitivity_qualitative/panels/cls_ha05/block_minus1 --num_samples 20 --method finercam --alpha 0.8 --panel_items rgb_gt_mask,gradcam_a,map_diff,finercam --mask_root /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000 --mask_col mask_rel_path --target_block_index -1 --clinician_labels --model_display_name 'CLS HA 0.5 block -1' --save_json --save_raw_cams --class_names MEL,NV --compare_mode gt_pair --

## 4. Build block sensitivity PDFs

This creates one PDF per model:
- `block_cam_sensitivity_cls_ha05_<mode>.pdf`
- `block_cam_sensitivity_gap_ha025_<mode>.pdf`

Each page is one image. Each row is one transformer block.

In [4]:
from PIL import Image, ImageDraw, ImageFont


def get_font(size: int, bold: bool = False):
    candidates = [
        "/System/Library/Fonts/Supplemental/Arial Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Arial.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    ]
    for path in candidates:
        if path and Path(path).exists():
            return ImageFont.truetype(path, size=size)
    return ImageFont.load_default()


FONT_TITLE = get_font(34, bold=True)
FONT_SUBTITLE = get_font(22, bold=False)
FONT_LABEL = get_font(22, bold=True)
FONT_SMALL = get_font(17, bold=False)


def image_id_to_stem(image_id_value: str) -> str:
    p = Path(str(image_id_value))
    stem = p.stem if p.suffix else p.name
    return stem.replace("/", "_").replace("\\", "_").replace(" ", "_")


def find_panel_png(scenario: dict, block_index: int, row: pd.Series) -> Path | None:
    out_dir = OUT_ROOT / "panels" / scenario["short_name"] / block_dir_name(block_index)
    candidates = []
    if "image_rel_path" in row and pd.notna(row["image_rel_path"]):
        candidates.append(image_id_to_stem(row["image_rel_path"]))
    if "image_id" in row and pd.notna(row["image_id"]):
        candidates.append(image_id_to_stem(row["image_id"]))
    if "image" in row and pd.notna(row["image"]):
        candidates.append(image_id_to_stem(row["image"]))

    for stem in dict.fromkeys(candidates):
        direct = out_dir / f"{stem}_{PANEL_SUFFIX}.png"
        if direct.exists():
            return direct
        matches = sorted(out_dir.glob(f"{stem}_*.png"))
        if matches:
            return matches[0]
    return None


def make_page_for_image_and_model(row: pd.Series, scenario: dict) -> Image.Image:
    page_width = 2200
    margin = 45
    label_width = 210
    gap = 14
    title_h = 115
    available_panel_width = page_width - 2 * margin - label_width - gap

    loaded_panels = []
    for block_index in TARGET_BLOCK_INDICES:
        panel_path = find_panel_png(scenario, block_index, row)
        if panel_path is None:
            loaded_panels.append((block_index, None, None))
            continue

        panel = Image.open(panel_path).convert("RGB")
        scale = available_panel_width / panel.width
        new_h = int(panel.height * scale)
        panel = panel.resize((available_panel_width, new_h), Image.Resampling.LANCZOS)
        loaded_panels.append((block_index, panel, panel_path))

    row_heights = [panel.height if panel is not None else 190 for _, panel, _ in loaded_panels]
    page_height = title_h + margin + sum(row_heights) + gap * (len(row_heights) - 1) + margin
    page = Image.new("RGB", (page_width, page_height), "white")
    draw = ImageDraw.Draw(page)

    image_id = row.get("image_id", row.get("image_rel_path", "unknown"))
    gt = row.get(GT_COL, row.get("gt_label", "unknown"))
    title = f"Block CAM sensitivity: {scenario['name']}"
    subtitle = f"Image: {image_id} | Ground truth: {gt} | Pooling: {scenario['pooling']} | Mode: {SAMPLE_MODE}"
    draw.text((margin, 26), title, fill="black", font=FONT_TITLE)
    draw.text((margin, 75), subtitle, fill=(60, 60, 60), font=FONT_SUBTITLE)

    y = title_h
    for block_index, panel, panel_path in loaded_panels:
        row_h = panel.height if panel is not None else 190
        label_x = margin
        label_y = y + 20
        draw.text((label_x, label_y), f"Block {block_index}", fill="black", font=FONT_LABEL)
        draw.text((label_x, label_y + 32), "qualitative", fill=(80, 80, 80), font=FONT_SMALL)

        if panel is None:
            box_x = margin + label_width + gap
            box_y = y
            draw.rectangle([box_x, box_y, box_x + available_panel_width, box_y + row_h], outline=(180, 180, 180), width=2)
            draw.text((box_x + 30, box_y + 65), "Missing panel PNG", fill=(160, 0, 0), font=FONT_LABEL)
        else:
            page.paste(panel, (margin + label_width + gap, y))
        y += row_h + gap

    return page


def build_pdf_for_scenario(scenario: dict):
    pages = []
    for _, row in DISPLAY_DF.iterrows():
        pages.append(make_page_for_image_and_model(row, scenario))

    if not pages:
        raise RuntimeError("No pages generated.")

    pdf_out = OUT_ROOT / f"block_cam_sensitivity_{scenario['short_name']}_{SAMPLE_MODE}.pdf"
    pages[0].save(pdf_out, save_all=True, append_images=pages[1:], resolution=150.0)
    print("Saved PDF:", pdf_out)
    return pdf_out


PDF_OUTPUTS = []
if DRY_RUN:
    print("DRY_RUN=True, skipping PDF build.")
else:
    for scenario in SCENARIOS:
        PDF_OUTPUTS.append(build_pdf_for_scenario(scenario))


Saved PDF: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/mel_nv/block_cam_sensitivity_qualitative/block_cam_sensitivity_cls_ha05_qualitative.pdf
Saved PDF: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/mel_nv/block_cam_sensitivity_qualitative/block_cam_sensitivity_gap_ha025_qualitative.pdf


## 5. Save config and quick checks


In [5]:
config = {
    "sample_mode": SAMPLE_MODE,
    "target_block_indices": TARGET_BLOCK_INDICES,
    "num_samples": NUM_SAMPLES,
    "active_csv": str(ACTIVE_CSV),
    "out_root": str(OUT_ROOT),
    "panel_items": PANEL_ITEMS,
    "alpha": ALPHA,
    "pdf_outputs": [str(p) for p in PDF_OUTPUTS],
    "scenarios": [
        {
            "name": s["name"],
            "short_name": s["short_name"],
            "checkpoint": str(s["checkpoint"]),
            "checkpoint_model_type": s.get("checkpoint_model_type", "panderm"),
            "pooling": s["pooling"],
        }
        for s in SCENARIOS
    ],
    "class_args": CLASS_ARGS,
    "compare_args": COMPARE_ARGS,
}

config_out = OUT_ROOT / f"block_cam_sensitivity_config_{SAMPLE_MODE}.json"
config_out.write_text(json.dumps(config, indent=2))
print("Saved config:", config_out)

for scenario in SCENARIOS:
    print("\n" + "=" * 80)
    print(scenario["name"])
    for block_index in TARGET_BLOCK_INDICES:
        out_dir = OUT_ROOT / "panels" / scenario["short_name"] / block_dir_name(block_index)
        pngs = sorted(out_dir.glob("*.png"))
        metas = sorted(out_dir.glob("*_meta.json"))
        raw_dir = out_dir / "raw_cams"
        raw_files = sorted(raw_dir.glob("*.npy")) if raw_dir.exists() else []
        print(f"  block {block_index:>3}: png={len(pngs):>4} meta={len(metas):>4} raw_flat={len(raw_files):>4}")

print("\nPDF outputs:")
for p in PDF_OUTPUTS:
    print(" ", p)


Saved config: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/mel_nv/block_cam_sensitivity_qualitative/block_cam_sensitivity_config_qualitative.json

CLS HA 0.5
  block  -1: png=  20 meta=  20 raw_flat=  80
  block  -2: png=  20 meta=  20 raw_flat=  80
  block  -4: png=  20 meta=  20 raw_flat=  80
  block  -6: png=  20 meta=  20 raw_flat=  80
  block  -8: png=  20 meta=  20 raw_flat=  80
  block -10: png=  20 meta=  20 raw_flat=  80
  block -12: png=  20 meta=  20 raw_flat=  80

GAP HA 0.25
  block  -1: png=  20 meta=  20 raw_flat=  80
  block  -2: png=  20 meta=  20 raw_flat=  80
  block  -4: png=  20 meta=  20 raw_flat=  80
  block  -6: png=  20 meta=  20 raw_flat=  80
  block  -8: png=  20 meta=  20 raw_flat=  80
  block -10: png=  20 meta=  20 raw_flat=  80
  block -12: png=  20 meta=  20 raw_flat=  80

PDF outputs:
  /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/mel_nv/block_cam_sensitivity_qualitative/block_cam_sensitivity_cls_ha05_qualitative.pdf
  /U

## 6. Practical interpretation checklist

When reviewing the PDFs, look for:

- Does the heatmap stay inside the lesion outline?
- Does it focus on lesion border, pigment network, dark/irregular areas, or clinically plausible structures?
- Does it focus on artifacts such as hair, dark corners, ruler marks, or background?
- Do later blocks become more concentrated or more diffuse?
- Does CLS behave differently from GAP?
- Which block gives stable maps across both MEL and NV examples?

Do not choose the final block only from this notebook. Use this to form hypotheses, then confirm quantitatively in notebook 03.